# Figure 2 Notebook — Part 2
## Vascular Cell Sub-clustering and Final Cell Type Annotation

**Input:** `healthycells_allsamples_no9317162_labeledCV.h5ad` → `adataf`

**Outputs:**
- `healthyvascularcells_allsamples_no9317162_labeledCV.h5ad` → `vasc_heathy`
- `healthycells_allsamples_no9317162_labeledCV.h5ad` → `adataf` (updated)
- `ND_vascells_allsamples_filteredlabeled.h5ad` → final saved vascular object

---

### Pipeline Overview

1. Subset mesenchymal + endothelial cells → `vasc_heathy`
2. Leiden clustering (res=1.0) on CONCORD embedding
3. Cluster QC: dotplots, marker UMAPs, Wilcoxon DEGs per cluster
4. Clusters 2 and 8 removed (ambiguous/low quality)
5. Remaining clusters merged into broad cell types
6. Cluster 10 re-clustered (res=0.5) to resolve heterogeneous population
7. Final labels assigned to `cell_type_final`
8. Islet-associated Fibroblasts defined by Location × cell type intersection
9. Vascular labels transferred back to full `adataf`
10. Remaining unlabeled mesenchymal cells re-clustered and reassigned
11. Final objects saved

---

### Cluster → Cell Type Mapping

| Leiden Clusters | Final Label |
|---|---|
| 0, 1, 3, 7, 11, 13 | Endothelial Cells |
| 4, 5, 6 | Fibroblasts |
| 9, 12, 14 | Pericytes |
| 10,4 | Pericytes |
| 10,0 / 10,1 / 10,2 / 10,3 / 10,5 | Islet-associated Fibroblasts |
| 10,6 | Removed |
| 2, 8 | Removed |
| Fibroblasts in Islet Location | → Islet-associated Fibroblasts |

**Note:** Islet-associated Fibroblasts are defined as fibroblasts
physically located within the islet compartment (Location == "Islet"),
confirmed by DEG analysis (Wilcoxon, FDR < 0.05).

---

### Key Output Columns

| Column | Object | Description |
|---|---|---|
| `vasc_leiden` | vasc_heathy | Raw Leiden clusters (res=1.0) |
| `cluster10_leiden` | vasc_heathy | Re-clustering of cluster 10 (res=0.5) |
| `cell_type` | vasc_heathy | Intermediate merged labels |
| `cell_type_final` | vasc_heathy | Final 4-category vascular labels |
| `merged_cell_type_refined` | adataf | Full dataset with refined vascular labels |

### Remaining Mesenchymal Reassignment
Cells labeled "Mesenchymal Cells" in `adataf` that were not captured
by `vasc_heathy` (edge cases outside the vascular subset) were
re-clustered at res=0.5 and reassigned:

| Leiden | Label |
|---|---|
| 0, 2 | Alpha Cells |
| 1 | Acinar Cells |
| 3 | Ductal Cells |
| 4 | Fibroblasts |
| 5 | Immune Cells |

---

### Color Palette
| Cell Type | Hex |
|---|---|
| Pericytes | #FD15FA |
| Islet-associated Fibroblasts | #f9a942 |
| Endothelial Cells | #08c93b |
| Fibroblasts | #0eecf8 |
| Islet | #3B0A45 |
| Exocrine | #CFA0E9 |

In [ ]:
# ══════════════════════════════════════════════════════════
# SECTION 1 — VASCULAR SUBSET + INITIAL LEIDEN CLUSTERING
# Subset mesenchymal + endothelial cells from adataf → vasc_heathy
# Leiden clustering at res=1.0 on CONCORD embedding
# Inspect clusters with marker gene UMAPs and dotplots
# ══════════════════════════════════════════════════════════

group_col = "merged_cell_type"

vasc_heathy = adata[
    adata.obs[group_col].isin(["Mesenchymal Cells", "Endothelial Cells"])
].copy()

In [ ]:
# SECTION 2 — CLUSTER QC + REMOVAL
# Clusters 2 and 8 removed based on:
#   - Low marker gene expression
#   - Mixed/ambiguous identity on dotplot
#   - Poor spatial coherence in UMAP
sc.pp.neighbors(vasc_heathy, use_rep='Concord')
sc.tl.leiden(vasc_heathy, resolution=1.0, key_added='vasc_leiden')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PLOT
# =========================
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAP",
    color=[
        "vasc_leiden",
        "Location",
        "RGS5",
        "PECAM1",
        "JUN",
        "CHGA"
    ],
    frameon=True,
    size=10,
    ncols=3,
    legend_loc="right margin",
    show=False,
    sort_order=True,
    cmap="cool",
    vmax=5
)

# =========================
# FORMAT
# =========================
fig = plt.gcf()
plt.rcParams["figure.figsize"] = (2, 2)
axes = fig.axes

for ax in axes:
    # clean title formatting
    title = ax.get_title()
    ax.set_title(
        title,
        fontsize=8,
        fontweight="bold"
    )

    # black borders
    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points
    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/"
    "UMAP_filtered_leiden_clusters"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PLOT
# =========================
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAP",
    color=[
        "LUM",
        "DCN",
        "CSPG4",
        "PLVAP",
        "COL1A1",
        "COL6A3"
    ],
    frameon=True,
    size=5,
    ncols=3,
    legend_loc="right margin",
    show=False,
    sort_order=True,
    cmap="cool",
    vmax=5
)

# =========================
# FORMAT
# =========================
fig = plt.gcf()
plt.rcParams["figure.figsize"] = (2, 2)

axes = fig.axes

for ax in axes:
    # clean title formatting
    title = ax.get_title()
    ax.set_title(
        title,
        fontsize=8,
        fontweight="bold"
    )

    # black borders
    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points
    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/"
    "UMAP_filtered_genes_vascleiden"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
vasc_heathy.obs['vasc_leiden'].value_counts()

In [ ]:
endo_peri_refined_markers = {
"ECs": ["PECAM1", "PLVAP", "VWF"],
"Pericytes": ["CSPG4", "PDGFRB", 'RGS5', 'DCN', "LUM"]
}

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 1)
# =========================
# DOTPLOT
# =========================
sc.pl.dotplot(
    vasc_heathy,
    endo_peri_refined_markers,
    groupby="vasc_leiden",
    vmax=5,
    show=False,
    swap_axes=True
)

# =========================
# SAVE HIGH RES
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/endo_peri_refined_dotplotascleiden.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/endo_peri_refined_dotplotascleiden.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/endo_peri_refined_dotplotvascleiden.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DOTPLOT
# =========================
sc.pl.dotplot(
    vasc_heathy,
    vasc_marker_genes_dict,
    groupby="vasc_leiden",
    vmax=5,
    show=False
)

# =========================
# SAVE HIGH RES
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/vascdict_dotplotascleiden.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/vascdict_dotplotascleiden.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Figure_2_plots/vascdict_dotplotvascleiden.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
sc.tl.rank_genes_groups(vasc_heathy, "vasc_leiden", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(vasc_heathy, n_genes=20)

In [ ]:
vasc_heathy = vasc_heathy[
    ~vasc_heathy.obs["vasc_leiden"].isin(["2", "8"])
].copy()

In [ ]:
sc.pl.embedding(
    vasc_heathy,
    basis='Concord_UMAP',
    color=['vasc_leiden', 'Location', 'RGS5', 'PECAM1'],
    frameon=False,
    ncols=3,
    vmax=5
)

In [ ]:
vasc_heathy.obs['vasc_leiden'].value_counts()

In [ ]:
sc.pl.dotplot(vasc_heathy, endo_peri_refined_markers, groupby='vasc_leiden', vmax=5)

In [ ]:
# SECTION 3 — MERGE CLUSTERS INTO BROAD CELL TYPES
# Leiden clusters merged into 4 cell types based on marker dotplots:
#   0,1,3,7,11,13 → Endothelial Cells
#   4,5,6         → Fibroblasts
#   9,12,14       → Pericytes
#   10            → ambiguous; re-clustered below
# Example usage:
# Define clusters to merge
merge_dict = {
    ('0', '1', '3', '7', '11', '13'): 'Endothelial Cells',
    ('4', '5', '6'): 'Fibroblasts',
    ('9', '12', '14'): 'Pericytes'
}

# Merge clusters
vasc_heathy = merge_clusters(vasc_heathy, cluster_column="vasc_leiden", merge_dict=merge_dict, new_column_name="cell_type")

# Visualize the updated clusters
print(vasc_heathy.obs["cell_type"].value_counts())

In [ ]:
# SECTION 4 — RE-CLUSTER CLUSTER 10
# Cluster 10 is heterogeneous — re-clustered at res=0.5
# to separate pericyte-like (10,4) from fibroblast-like (10,0–10,5)
# Cluster 10,6 removed (contamination/low quality)
# Result stored in cell_type_final
vasc_heathy.obs["cell_type"] = (
    vasc_heathy.obs["cell_type"]
    .astype("category")
)

# recluster only cluster 10
sc.tl.leiden(
    vasc_heathy,
    resolution=0.5,
    restrict_to=("cell_type", ["10"]),
    key_added="cluster10_leiden"
)

In [ ]:
sc.pl.dotplot(vasc_heathy, endo_peri_refined_markers, groupby='cluster10_leiden', vmax=5)

In [ ]:
vasc_heathy.obs['cluster10_leiden'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(vasc_heathy, "cluster10_leiden", use_raw=False, layer='counts', method="wilcoxon")
sc.pl.rank_genes_groups(vasc_heathy, n_genes=20)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# make text larger globally
mpl.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

# run DE
sc.tl.rank_genes_groups(
    vasc_heathy,
    "cluster10_leiden",
    use_raw=False,
    method="wilcoxon"
)

# MUCH larger figure
sc.pl.rank_genes_groups(
    vasc_heathy,
    n_genes=25,
    figsize=(24, 16),
    sharey=False
)

plt.show()

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    vasc_heathy,
    n_genes=8,
    standard_scale="var",
    figsize=(16, 4)
)

In [ ]:
# make editable copy
vasc_heathy.obs["cell_type_final"] = vasc_heathy.obs["cluster10_leiden"].astype(str)

# merge 10,4 into Pericytes
vasc_heathy.obs.loc[
    vasc_heathy.obs["cluster10_leiden"] == "10,4",
    "cell_type_final"
] = "Pericytes"

# remove 10,6 cells
vasc_heathy = vasc_heathy[
    vasc_heathy.obs["cluster10_leiden"] != "10,6"
].copy()

# merge ALL remaining 10,x clusters into Fibroblasts
clusters_to_merge = [
    "10,0",
    "10,1",
    "10,2",
    "10,3",
    "10,5"
]

vasc_heathy.obs.loc[
    vasc_heathy.obs["cluster10_leiden"].isin(clusters_to_merge),
    "cell_type_final"
] = "Islet-associated Fibroblasts"

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# CUSTOM CELL PALETTE
# =========================



# =========================
# PLOT
# =========================
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAP",
    color=["Sample"],
    frameon=False,
    ncols=2,
    size=10,
    show=False
)

# =========================
# FORMAT
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:

    title = ax.get_title()

    ax.set_title(
        title,
        fontsize=8,
        fontweight="bold"
    )

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points
    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/"
    "UMAP_labeled_vascleiden_donorid"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# SECTION 5 — RECOMPUTE UMAP ON VASCULAR SUBSET
# Run fresh CONCORD UMAP (Concord_UMAPmRU) on vascular cells only
# cosine metric, n_neighbors=30 to match full dataset parameters
# Used for all final vascular figure panels

ccd.ul.run_umap(vasc_heathy, source_key='Concord', result_key='Concord_UMAPmRU', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['cell_type', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    vasc_heathy, basis='Concord_UMAPmRU', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
color_by = ['cell_type_final', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    vasc_heathy, basis='Concord_UMAPmRU', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PALETTES
# =========================
cell_palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b",
    "Fibroblasts": "#0eecf8"
}

location_palette = {
    "Islet": "#3B0A45",      # dark purple
    "Exocrine": "#CFA0E9"    # light purple
}

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(1, 2, figsize=(4, 2))

# ---- PANEL 1: CELL TYPE ----
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAPmRU",
    color="cell_type_final",
    palette=cell_palette,
    size=5,
    frameon=False,
    legend_loc="center left",
    ax=axes[0],
    show=False,
    sort_order=True
)

# ---- PANEL 2: LOCATION ----
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAPmRU",
    color="Location",
    palette=location_palette,
    size=10,
    frameon=False,
    legend_loc="center left",
    ax=axes[1],
    show=False,
    sort_order=True
)

# =========================
# FORMAT
# =========================
for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points for small file size
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_vasc_celltype_location"

os.makedirs(os.path.dirname(out_base), exist_ok=True)

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")

plt.show()

print("✔ DONE — clean UMAP exported")

In [ ]:
# SECTION 6 — ISLET-ASSOCIATED FIBROBLAST DEFINITION
# "Islet-associated Fibroblasts" defined as fibroblasts
# physically located in the Islet compartment (Location == "Islet")
# Confirmed by DEG: Islet vs Exocrine fibroblasts (Wilcoxon, FDR < 0.05)
# Fibroblasts in Exocrine compartment reverted to "Fibroblasts"
# Cell counts verified per donor before and after reassignment
# COUNT FIBROBLASTS IN ISLET REGION
# ==========================================

CELLTYPE_COL = "cell_type_final"   # change if needed
LOCATION_COL = "Location"          # change if needed

# ------------------------------------------
# Subset fibroblasts in islets
# ------------------------------------------

fibro_islet = vasc_heathy.obs[
    (vasc_heathy.obs[CELLTYPE_COL] == "Fibroblasts") &
    (vasc_heathy.obs[LOCATION_COL] == "Islet")
]

# ------------------------------------------
# Print total count
# ------------------------------------------

print("Number of fibroblasts in islets:")
print(len(fibro_islet))

# ------------------------------------------
# Optional: counts per donor
# ------------------------------------------

fibro_per_donor = (
    fibro_islet["Sample"]
    .value_counts()
    .sort_index()
)

print("\nFibroblasts in islets per donor:")
print(fibro_per_donor)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# =====================================================
# SETTINGS
# =====================================================
CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"

FIBRO_LABEL = "Fibroblasts"

# =====================================================
# SUBSET FIBROBLASTS ONLY
# =====================================================
fibroblasts = vasc_heathy[
    vasc_heathy.obs[CELLTYPE_COL] == FIBRO_LABEL
].copy()

print(fibroblasts)

# =====================================================
# CHECK GROUP SIZES
# =====================================================
print("\nFibroblasts by location:")
print(
    fibroblasts.obs[LOCATION_COL]
    .value_counts()
)

# =====================================================
# OPTIONAL:
# REMOVE VERY LOWLY DETECTED GENES
# =====================================================
sc.pp.filter_genes(fibroblasts, min_cells=5)

# =====================================================
# DEG:
# ISLET vs EXOCRINE FIBROBLASTS
# =====================================================
sc.tl.rank_genes_groups(
    fibroblasts,
    groupby=LOCATION_COL,
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    pts=True,
    use_raw=False
)

# =====================================================
# EXTRACT RESULTS
# =====================================================
deg = sc.get.rank_genes_groups_df(
    fibroblasts,
    group="Islet"
)

# =====================================================
# ADD SIMPLE FILTERS
# =====================================================
deg = deg[
    (deg["pvals_adj"] < 0.05)
]

# sort
deg = deg.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# SAVE CSV
# =====================================================
outpath = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_Islet_vs_Exocrine_Fibroblasts.csv"

deg.to_csv(outpath, index=False)

# =====================================================
# SHOW TOP GENES
# =====================================================
print("\nTOP ENRICHED IN ISLET FIBROBLASTS:")
print(
    deg[
        ["names", "logfoldchanges", "pvals_adj"]
    ].head(30)
)

print(f"\nSaved to:\n{outpath}")

In [ ]:
# ==========================================
# MERGE ONLY ISLET FIBROBLASTS
# INTO "Islet-associated Fibroblasts"
# ==========================================

CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"

# ------------------------------------------
# Convert to string temporarily
# ------------------------------------------

vasc_heathy.obs[CELLTYPE_COL] = (
    vasc_heathy.obs[CELLTYPE_COL]
    .astype(str)
)

# ------------------------------------------
# Rename ONLY fibroblasts located in islets
# ------------------------------------------

vasc_heathy.obs.loc[
    (vasc_heathy.obs[CELLTYPE_COL] == "Fibroblasts") &
    (vasc_heathy.obs[LOCATION_COL] == "Islet"),
    CELLTYPE_COL
] = "Islet-associated Fibroblasts"

# ------------------------------------------
# Convert back to category
# ------------------------------------------

vasc_heathy.obs[CELLTYPE_COL] = (
    vasc_heathy.obs[CELLTYPE_COL]
    .astype("category")
)

# ------------------------------------------
# Check counts
# ------------------------------------------

print(
    vasc_heathy.obs[CELLTYPE_COL]
    .value_counts()
)

In [ ]:
# ==========================================
# COUNT FIBROBLASTS IN ISLET REGION
# ==========================================

CELLTYPE_COL = "cell_type_final"   # change if needed
LOCATION_COL = "Location"          # change if needed

# ------------------------------------------
# Subset fibroblasts in islets
# ------------------------------------------

fibro_islet = vasc_heathy.obs[
    (vasc_heathy.obs[CELLTYPE_COL] == "Fibroblasts") &
    (vasc_heathy.obs[LOCATION_COL] == "Islet")
]

# ------------------------------------------
# Print total count
# ------------------------------------------

print("Number of fibroblasts in islets:")
print(len(fibro_islet))

# ------------------------------------------
# Optional: counts per donor
# ------------------------------------------

fibro_per_donor = (
    fibro_islet["Sample"]
    .value_counts()
    .sort_index()
)

print("\nFibroblasts in islets per donor:")
print(fibro_per_donor)

In [ ]:
# ==========================================
# COUNT FIBROBLASTS IN ISLET REGION
# ==========================================

CELLTYPE_COL = "cell_type_final"   # change if needed
LOCATION_COL = "Location"          # change if needed

# ------------------------------------------
# Subset fibroblasts in islets
# ------------------------------------------

fibro_islet = vasc_heathy.obs[
    (vasc_heathy.obs[CELLTYPE_COL] == "Islet-associated Fibroblasts") &
    (vasc_heathy.obs[LOCATION_COL] == "Exocrine")
]

# ------------------------------------------
# Print total count
# ------------------------------------------

print("Number of Islet-associated fibroblasts in exocrine:")
print(len(fibro_islet))

# ------------------------------------------
# Optional: counts per donor
# ------------------------------------------

fibro_per_donor = (
    fibro_islet["Sample"]
    .value_counts()
    .sort_index()
)

print("\nFibroblasts in exo per donor:")
print(fibro_per_donor)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# =====================================================
# SETTINGS
# =====================================================
CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"

FIBRO_LABEL = "Islet-associated Fibroblasts"

# =====================================================
# SUBSET FIBROBLASTS ONLY
# =====================================================
islet_fibroblasts = vasc_heathy[
    vasc_heathy.obs[CELLTYPE_COL] == FIBRO_LABEL
].copy()

print(islet_fibroblasts)

# =====================================================
# CHECK GROUP SIZES
# =====================================================
print("\nFibroblasts by location:")
print(
    islet_fibroblasts.obs[LOCATION_COL]
    .value_counts()
)

# =====================================================
# OPTIONAL:
# REMOVE VERY LOWLY DETECTED GENES
# =====================================================
sc.pp.filter_genes(islet_fibroblasts, min_cells=5)

# =====================================================
# DEG:
# ISLET vs EXOCRINE FIBROBLASTS
# =====================================================
sc.tl.rank_genes_groups(
    islet_fibroblasts,
    groupby=LOCATION_COL,
    groups=["Exocrine"],
    reference="Islet",
    method="wilcoxon",
    pts=True,
    use_raw=False
)

# =====================================================
# EXTRACT RESULTS
# =====================================================
deg = sc.get.rank_genes_groups_df(
    islet_fibroblasts,
    group="Exocrine"
)

# =====================================================
# ADD SIMPLE FILTERS
# =====================================================
deg = deg[
    (deg["pvals_adj"] < 0.05)
]

# sort
deg = deg.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# SAVE CSV
# =====================================================
outpath = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_Islet_vs_Exocrine_isletFibroblasts.csv"

deg.to_csv(outpath, index=False)

# =====================================================
# SHOW TOP GENES
# =====================================================
print("\nTOP ENRICHED IN ISLET FIBROBLASTS:")
print(
    deg[
        ["names", "logfoldchanges", "pvals_adj"]
    ].head(30)
)

print(f"\nSaved to:\n{outpath}")

In [ ]:
# ==========================================
# MERGE ONLY ISLET FIBROBLASTS
# INTO "Islet-associated Fibroblasts"
# ==========================================

CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"

# ------------------------------------------
# Convert to string temporarily
# ------------------------------------------

vasc_heathy.obs[CELLTYPE_COL] = (
    vasc_heathy.obs[CELLTYPE_COL]
    .astype(str)
)

# ------------------------------------------
# Rename ONLY fibroblasts located in islets
# ------------------------------------------

vasc_heathy.obs.loc[
    (vasc_heathy.obs[CELLTYPE_COL] == "Islet-associated Fibroblasts") &
    (vasc_heathy.obs[LOCATION_COL] == "Exocrine"),
    CELLTYPE_COL
] = "Fibroblasts"

# ------------------------------------------
# Convert back to category
# ------------------------------------------

vasc_heathy.obs[CELLTYPE_COL] = (
    vasc_heathy.obs[CELLTYPE_COL]
    .astype("category")
)

# ------------------------------------------
# Check counts
# ------------------------------------------

print(
    vasc_heathy.obs[CELLTYPE_COL]
    .value_counts()
)

In [ ]:
# ==========================================
# COUNT FIBROBLASTS IN ISLET REGION
# ==========================================

CELLTYPE_COL = "cell_type_final"   # change if needed
LOCATION_COL = "Location"          # change if needed

# ------------------------------------------
# Subset fibroblasts in islets
# ------------------------------------------

fibro_islet = vasc_heathy.obs[
    (vasc_heathy.obs[CELLTYPE_COL] == "Islet-associated Fibroblasts") &
    (vasc_heathy.obs[LOCATION_COL] == "Exocrine")
]

# ------------------------------------------
# Print total count
# ------------------------------------------

print("Number of fibroblasts in islets:")
print(len(fibro_islet))

# ------------------------------------------
# Optional: counts per donor
# ------------------------------------------

fibro_per_donor = (
    fibro_islet["Sample"]
    .value_counts()
    .sort_index()
)

print("\nFibroblasts in islets per donor:")
print(fibro_per_donor)

In [ ]:
color_map = {
    "Islet": "#008080",     # yellow
    "Exocrine": "#FFD700"   # teal
}



# -------------------------
# PLOT
# -------------------------
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAP",
    color=["Location", "vasc_leiden"],
    size=10,
    ncols=2,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)


# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_filtered_locationsamples_vasc"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PALETTES
# =========================
cell_palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b",
    "Fibroblasts": "#0eecf8"
}

location_palette = {
    "Islet": "#3B0A45",      # dark purple
    "Exocrine": "#CFA0E9"    # light purple
}

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(1, 2, figsize=(4, 2))

# ---- PANEL 1: CELL TYPE ----
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAPmRU",
    color="cell_type_final",
    palette=cell_palette,
    size=5,
    frameon=False,
    legend_loc="center left",
    ax=axes[0],
    show=False,
    sort_order=True
)

# ---- PANEL 2: LOCATION ----
sc.pl.embedding(
    vasc_heathy,
    basis="Concord_UMAPmRU",
    color="Location",
    palette=location_palette,
    size=5,
    frameon=False,
    legend_loc="center left",
    ax=axes[1],
    show=False,
    sort_order=True
)

# =========================
# FORMAT
# =========================
for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points for small file size
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_vasc_celltype_location"

os.makedirs(os.path.dirname(out_base), exist_ok=True)

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")

plt.show()

print("✔ DONE — clean UMAP exported")

In [ ]:
vasc_heathy.write(
    "/Users/kmeneses/Desktop/updated_data/"
    "healthyvascularcells_allsamples_no9317162_labeledCV.h5ad"
)

In [ ]:
# SECTION 7 — TRANSFER LABELS TO FULL DATASET
# Transfer cell_type_final from vasc_heathy → adataf.obs["merged_cell_type_refined"]
# Remaining "Mesenchymal Cells" in adataf not covered by vasc_heathy
# are re-clustered (res=0.5) and reassigned to:
#   Alpha Cells, Acinar Cells, Ductal Cells, Fibroblasts, Immune Cells
# Final column: adataf.obs["merged_cell_type_refined"]

In [ ]:
# make new column as STRING instead of categorical
adataf.obs["merged_cell_type_refined"] = (
    adataf.obs["merged_cell_type"].astype(str)
)

# transfer refined vascular labels
adataf.obs.loc[
    vasc_heathy.obs_names,
    "merged_cell_type_refined"
] = vasc_heathy.obs["cell_type_final"].astype(str).values

# optional: convert back to category
adataf.obs["merged_cell_type_refined"] = (
    adataf.obs["merged_cell_type_refined"].astype("category")
)

In [ ]:
len(set(adataf.obs_names) & set(vasc_heathy.obs_names))

In [ ]:
vasc_heathy.obs["cell_type_final"].value_counts()

In [ ]:
adataf.obs[
    adataf.obs["merged_cell_type_refined"] == "Mesenchymal Cells"
].head()

In [ ]:
adataf.obs['merged_cell_type_refined'].value_counts()

In [ ]:
# subset remaining unlabeled mesenchymal cells
remaining_mes = adataf[
    adataf.obs["merged_cell_type_refined"] == "Mesenchymal Cells"
].copy()

# neighbors + UMAP if needed
sc.pp.neighbors(remaining_mes, use_rep="Concord")
sc.tl.umap(remaining_mes)

# Leiden subclustering
sc.tl.leiden(
    remaining_mes,
    resolution=0.5,
    key_added="remaining_mes_leiden"
)

# visualize clusters
sc.pl.umap(
    remaining_mes,
    color=["remaining_mes_leiden"]
)

# run DE
sc.tl.rank_genes_groups(
    remaining_mes,
    groupby="remaining_mes_leiden",
    method="wilcoxon",
    use_raw=False
)

# inspect markers
sc.pl.rank_genes_groups_dotplot(
    remaining_mes,
    n_genes=8,
    standard_scale="var"
)

# Transfer vasc_heathy cell_type_final labels → adataf
 Remaining "Mesenchymal Cells" in adataf re-clustered and
 reassigned to correct broad cell types (alpha, ductal, etc.)
 Final refined column: adataf.obs["merged_cell_type_refined"]

In [ ]:
# rename clusters directly in full AnnData

rename_dict = {
    "2": "Alpha Cells",
    "4": "Fibroblasts",
    "3": "Ductal Cells",
    "1": "Acinar Cells",
    "0": "Alpha Cells",
    "5": "Immune Cells",
}

# map labels from remaining_mes back to full object
adataf.obs.loc[
    remaining_mes.obs_names,
    "merged_cell_type_refined"
] = (
    remaining_mes.obs["remaining_mes_leiden"]
    .astype(str)
    .map(rename_dict)
    .values
)

In [ ]:
adataf.obs['merged_cell_type_refined'].value_counts()

In [ ]:
# remove unused categories (including Mesenchymal Cells)
adataf.obs["merged_cell_type_refined"] = (
    adataf.obs["merged_cell_type_refined"]
    .cat.remove_unused_categories()
)

In [ ]:
adataf.obs['merged_cell_type_refined'].value_counts()

In [ ]:
color_by = ['merged_cell_type_refined', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adataf, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
adataf.write_h5ad('/Users/kmeneses/Desktop/updated_data/healthycells_allsamples_no9317162_labeledCV.h5ad')

In [ ]:
vasc_heathy.write_h5ad('/Users/kmeneses/Desktop/updated_data/healthyvascularcells_allsamples_no9317162_labeledCV.h5ad')

In [ ]:
vasc_heathy

In [ ]:
adataf

In [ ]:
vasc_heathy.write('/Users/kmeneses/Desktop/updated_data/ND_vascells_allsamples_filteredlabeled.h5ad')